J'importe pandas et numpy

In [2]:
import numpy as np

In [ ]:
import pandas as pd

Téléchargeons le fichier

In [ ]:
data=pd.read_excel('GL COLRUYT 2025.xlsx')

In [ ]:
data1=pd.read_excel('GL 2026 Colruyt.xlsx')

Verifions si le fichier est bien telecharger 

In [ ]:
data.head()

,N°compte,Mois,Date,Code journaux,N° pièces,libellés,Lettrage,Débit,Crédit,Solde cumulé,Solde au 31 12 2025
0,10130000,1,2025-01-01,RAN,1,A.N. au 010125,NaN,NaN,200000000.0,-200000000.0,-200000000
1,1110100,6,2025-06-30,OD,797,Affectation du résultat,NaN,NaN,668605.0,-668605.0,-668605
2,12000000,6,2025-06-30,OD,797,Affectation du résultat,NaN,NaN,4196400.0,-4196400.0,-4196400
3,12900000,1,2025-01-01,RAN,1,A.N. au 010125,NaN,1821048.0,NaN,1821048.0,1821048
4,12900000,6,2025-06-30,OD,797,Affectation du résultat,NaN,NaN,1821048.0,NaN,-1821048


In [ ]:
data.tail()

,N°compte,Mois,Date,Code journaux,N° pièces,libellés,Lettrage,Débit,Crédit,Solde cumulé,Solde au 31 12 2025
4039,7588000,2,2025-02-28,BQ1,219,Virement DGID,NaN,NaN,1.0,-11.0,-1
4040,7591000,10,2025-10-31,OD,1159,Reprise provision,NaN,NaN,5914953.0,-5914953.0,-5914953
4041,781000,12,2025-12-31,OD,1502,Transfert de charge,NaN,NaN,5009280.0,-5009280.0,-5009280
4042,7911000,10,2025-10-31,OD,1160,Reprise provision retraite,NaN,NaN,9220113.0,-9220113.0,-9220113
4043,8950000,12,2025-12-31,OD,1639,IMF,NaN,5000000.0,NaN,5000000.0,5000000


Regardons les infos du fichier

In [ ]:
data.info

<bound method DataFrame.info of       N°compte  Mois        Date Code journaux  N° pièces  \
0     10130000      1 2025-01-01           RAN          1   
1      1110100      6 2025-06-30            OD        797   
2     12000000      6 2025-06-30            OD        797   
3     12900000      1 2025-01-01           RAN          1   
4     12900000      6 2025-06-30            OD        797   
...        ...    ...        ...           ...        ...   
4039   7588000      2 2025-02-28           BQ1        219   
4040   7591000     10 2025-10-31            OD       1159   
4041    781000     12 2025-12-31            OD       1502   
4042   7911000     10 2025-10-31            OD       1160   
4043   8950000     12 2025-12-31            OD       1639   

                        libellés Lettrage       Débit       Crédit  \
0                 A.N. au 010125       NaN        NaN  200000000.0   
1        Affectation du résultat       NaN        NaN     668605.0   
2        Affectation du r

verifions s'il y a pas de valeurs dupliquées 

In [ ]:
data.duplicated().sum()

np.int64(0)

y'en a zero

on va générer un résumé statistique complet de notre DataFrame.

data.describe(include="all")

Faisons la somme des valeurs manquantes

In [ ]:
data.isnull().sum() 

N°compte                  0
Mois                      0
Date                      0
Code journaux             0
N° pièces                 0
libellés                  1
Lettrage               2944
Débit                  1884
Crédit                 2160
Solde cumulé            210
Solde au 31 12 2025       0
dtype: int64

On voit quand on additionne le crédit et le débit le total fait 4044 ce qui prouve qu'il n'y a pas de valeurs manquantes juste qu'en comptabilité c'est soit un crédit soit un débit rarement les deux

on va remplacer les NaN par 0 dans credit et debit

In [ ]:
data['Débit'] = data['Débit'].fillna(0)
data['Crédit']= data['Crédit'].fillna(0)


on va remplacer les NaN par non lettre dans lettrage

In [ ]:
data['Lettrage ']= data['Lettrage '].fillna("non lettre")

ici nous allons convertir les numeros de comptes en chaines de caracteres pour eviter les notations trop lourdes

In [ ]:
data['N°compte'] = data['N°compte'].astype(str)

le solde cumule ne peut pas etre un NaN car c'est une operation mathematique donc nous allons mettre sa formule pour y remedier

In [ ]:
data['Solde cumulé'] = (data['Débit'] - data['Crédit']).cumsum()

verifions si ca a marcher

In [ ]:
data.isnull().sum()


N°compte               0
Mois                   0
Date                   0
Code journaux          0
N° pièces              0
libellés               1
Lettrage               0
Débit                  0
Crédit                 0
Solde cumulé           0
Solde au 31 12 2025    0
dtype: int64

Y a un libellé manquant notifions ca 

In [ ]:
data['libellés'] = data['libellés'].fillna("Libellé manquant")

verifions tous ca 

In [ ]:
data.isnull().sum() 

N°compte               0
Mois                   0
Date                   0
Code journaux          0
N° pièces              0
libellés               0
Lettrage               0
Débit                  0
Crédit                 0
Solde cumulé           0
Solde au 31 12 2025    0
dtype: int64

commencons les extractions

1-	Agréger les grands livres obtenus et établir des totaux par compte : S’assurer qu’il n’existe pas d’écart. 



In [ ]:
# ÉTAPE 1 : On regroupe par 'N°compte' et on additionne les Débits et Crédits
tache1 = data.groupby('N°compte')[['Débit', 'Crédit']].sum().reset_index()

In [ ]:
# 2. Calcul du solde (Débit - Crédit)
tache1['Solde_calcule'] = tache1['Débit'] - tache1['Crédit']

In [ ]:
# 3. On récupère le solde au 31/12/2025 de référence pour chaque compte
soldes_ref = data.groupby('N°compte')['Solde cumulé'].last().reset_index()

In [ ]:
# 4. On fusionne pour pouvoir comparer
tache1 = pd.merge(tache1, soldes_ref, on='N°compte')

In [ ]:
# 5. On calcule l'écart
tache1['Ecart'] = tache1['Solde_calcule'] - tache1['Solde cumulé']

In [ ]:
# 6. On vérifie s'il y a des écarts
ecarts = tache1[tache1['Ecart'] != 0]

In [ ]:
if ecarts.empty:
    print("Aucun écart détecté ! Les totaux sont parfaits.")
else:
    print(f" {len(ecarts)} compte(s) avec écart(s) trouvé(s) :")
    display(ecarts)

 128 compte(s) avec écart(s) trouvé(s) :


,N°compte,Débit,Crédit,Solde_calcule,Solde cumulé,Ecart
1,1110100,0.0,668605.0,-668605.0,-200668605.0,200000000.0
2,12000000,0.0,4196400.0,-4196400.0,-204865005.0,200668605.0
3,12900000,1821048.0,1821048.0,0.0,-204865005.0,204865005.0
4,1301000,6686053.0,6686053.0,0.0,-204865005.0,204865005.0
5,1961000,9220113.0,19862674.0,-10642561.0,-215507566.0,204865005.0
...,...,...,...,...,...,...
124,7588000,0.0,11.0,-11.0,15144346.0,-15144357.0
125,7591000,0.0,5914953.0,-5914953.0,9229393.0,-15144346.0
126,781000,0.0,5009280.0,-5009280.0,4220113.0,-9229393.0
127,7911000,0.0,9220113.0,-9220113.0,-5000000.0,-4220113.0


In [ ]:
tache1.head()

,N°compte,Débit,Crédit,Solde_calcule,Solde cumulé,Ecart
0,10130000,0.0,200000000.0,-200000000.0,-200000000.0,0.0
1,1110100,0.0,668605.0,-668605.0,-200668605.0,200000000.0
2,12000000,0.0,4196400.0,-4196400.0,-204865005.0,200668605.0
3,12900000,1821048.0,1821048.0,0.0,-204865005.0,204865005.0
4,1301000,6686053.0,6686053.0,0.0,-204865005.0,204865005.0


2-	Écritures déséquilibrées : A partir du journal des écritures, extraire les écritures concernant les comptes de comptabilité générale repris dans la balance, ensuite faire un subtotal par numéro de pièce. Ensuite, pour chaque n° de pièce, s’assurer que le cumul au débit était égal au cumul au crédit.

In [ ]:
# 1. Subtotal par numéro de pièce (Somme des Débits et Crédits)
tache2 = data.groupby('N° pièces')[['Débit', 'Crédit']].sum().reset_index()

In [ ]:
# 2. Calcul de la différence entre Débit et Crédit pour chaque pièce
tache2['Ecart_piece'] = tache2['Débit'] - tache2['Crédit']

In [ ]:
# 3. Extraction des pièces dont le solde/écart n'est PAS égal à 0
ecritures_desequilibrees = tache2[tache2['Ecart_piece'] != 0]

In [ ]:
# 4. Affichage du résultat
if ecritures_desequilibrees.empty:
    print(" Aucune écriture déséquilibrée ! Toutes les pièces comptables respectent la partie double.")
else:
    print(f" Attention : {len(ecritures_desequilibrees)} pièce(s) déséquilibrée(s) trouvée(s) !")
    display(ecritures_desequilibrees)

 Attention : 131 pièce(s) déséquilibrée(s) trouvée(s) !


,N° pièces,Débit,Crédit,Ecart_piece
0,1,835966058.0,884648679.0,-48682621.0
1,2,55368674.0,6686053.0,48682621.0
3,4,350760.0,354000.0,-3240.0
20,21,526140.0,531000.0,-4860.0
25,27,602600.0,608000.0,-5400.0
...,...,...,...,...
1494,1507,58721970.0,59619133.0,-897163.0
1523,1536,441866.0,506622.0,-64756.0
1543,1556,2065363.0,1168200.0,897163.0
1546,1559,1504500.0,1504000.0,500.0


3-	Ecritures avec informations manquantes : A partir du grand livre extraire toutes les écritures ayant des informations manquantes.

In [ ]:
# 1. On identifie les cas où des informations essentielles sont manquantes ou génériques
condition_manquant = (
    (data['libellés'] == "Libellé manquant") |
    (data['Lettrage '] == "non lettre") |
    (data['Code journaux'].isnull()) |
    (data['N° pièces'].isnull()) |
    (data['Date'].isnull())
)

In [ ]:
# 2. On extrait l'ensemble des lignes concernées
ecritures_manquantes = data[condition_manquant]

In [ ]:
# 3. Affichage du résultat
print(f" Nombre d'écritures avec informations manquantes/non lettrées : {len(ecritures_manquantes)}")
display(ecritures_manquantes)

 Nombre d'écritures avec informations manquantes/non lettrées : 2944


,N°compte,Mois,Date,Code journaux,N° pièces,libellés,Lettrage,Débit,Crédit,Solde cumulé,Solde au 31 12 2025
0,10130000,1,2025-01-01,RAN,1,A.N. au 010125,non lettre,0.0,200000000.0,-200000000.0,-200000000
1,1110100,6,2025-06-30,OD,797,Affectation du résultat,non lettre,0.0,668605.0,-200668605.0,-668605
2,12000000,6,2025-06-30,OD,797,Affectation du résultat,non lettre,0.0,4196400.0,-204865005.0,-4196400
3,12900000,1,2025-01-01,RAN,1,A.N. au 010125,non lettre,1821048.0,0.0,-203043957.0,1821048
4,12900000,6,2025-06-30,OD,797,Affectation du résultat,non lettre,0.0,1821048.0,-204865005.0,-1821048
...,...,...,...,...,...,...,...,...,...,...,...
4039,7588000,2,2025-02-28,BQ1,219,Virement DGID,non lettre,0.0,1.0,15144346.0,-1
4040,7591000,10,2025-10-31,OD,1159,Reprise provision,non lettre,0.0,5914953.0,9229393.0,-5914953
4041,781000,12,2025-12-31,OD,1502,Transfert de charge,non lettre,0.0,5009280.0,4220113.0,-5009280
4042,7911000,10,2025-10-31,OD,1160,Reprise provision retraite,non lettre,0.0,9220113.0,-5000000.0,-9220113


4-	Identification des écritures enregistrées à la fin de l’exercice ou en tant qu’écritures post-clôture, comportant peu ou pas d’explication ou de description : A partir du fichier des écritures, sélectionner toutes les écritures qui ont une date comptable comprise entre le 20/12/2025 et le 31 décembre 2025 Identification des écritures effectuées sur des comptes non reliés, inhabituels ou rarement utilisés



In [ ]:
#Filtrer les écritures de fin d'année (entre le 20/12/2025 et le 31/12/2025)
fin_annee = data[(data['Date'] >= '2025-12-20') & (data['Date'] <= '2025-12-31')]

In [ ]:
# Récupérer la liste unique des comptes mouvementés sur cette période de fin d'année
comptes_fin_annee = fin_annee['N°compte'].unique()

In [ ]:
# ÉTAPE 3 : Vérifier le volume total de transactions sur l'année pour ces comptes
resultats_suspects = []

for compte in comptes_fin_annee:
    # Compter le nombre total de lignes (transactions) pour ce compte sur l'ensemble de l'année
    nb_total_transactions_annee = len(data[data['N°compte'] == compte])
    
    # Si le compte a moins de 5 transactions sur l'année au total, il est suspect
    if nb_total_transactions_annee < 5:
        resultats_suspects.append({
            'N°compte': compte,
            'Nombre_total_transactions_annee': nb_total_transactions_annee
        })

In [ ]:
df_suspects = pd.DataFrame(resultats_suspects)

In [ ]:
if df_suspects.empty:
    print("Aucun compte suspect trouvé (tous les comptes mouvementés en fin d'année ont au moins 5 transactions sur l'année).")
else:
    print(f"Attention : {len(df_suspects)} compte(s) suspect(s) identifié(s) (mouvementés en fin d'année avec moins de 5 transactions au total sur l'année) :")
    display(df_suspects)
    
    print("\nDétail des écritures de fin d'année pour ces comptes suspects :")
    display(fin_annee[fin_annee['N°compte'].isin(df_suspects['N°compte'])])

Attention : 11 compte(s) suspect(s) identifié(s) (mouvementés en fin d'année avec moins de 5 transactions au total sur l'année) :


,N°compte,Nombre_total_transactions_annee
0,1961000,3
1,2844000,2
2,28451000,2
3,31110100,2
4,401105100,4
5,4991000,3
6,6591000,1
7,6813000,1
8,69110000,1
9,781000,1



Détail des écritures de fin d'année pour ces comptes suspects :


,N°compte,Mois,Date,Code journaux,N° pièces,libellés,Lettrage,Débit,Crédit,Solde cumulé,Solde au 31 12 2025
9,1961000,12,2025-12-31,OD,1506,Provisions pour retraite,non lettre,0.0,10642561.0,-2.155076e+08,-10642561
46,2844000,12,2025-12-31,OD,1505,Dotation aux amortissements,non lettre,0.0,1121442.0,-1.812688e+08,-1121442
48,28451000,12,2025-12-31,OD,1505,Dotation aux amortissements,non lettre,0.0,5262195.0,-2.024021e+08,-5262195
50,31110100,12,2025-12-31,OD,1636,Solde stock initial,non lettre,0.0,2256800.0,-2.024021e+08,-2256800
494,401105100,12,2025-12-24,BQ1,1561,Virement,non lettre,1189440.0,0.0,-6.043405e+08,1189440
1289,4991000,12,2025-12-31,OD,1638,Provisons pour congés,non lettre,0.0,3189280.0,-9.769010e+07,-3189280
3896,6591000,12,2025-12-31,OD,1638,Provisons pour congés,non lettre,3189280.0,0.0,1.280742e+09,3189280
3948,6813000,12,2025-12-31,OD,1505,Dotation aux amortissements,non lettre,6383637.0,0.0,1.460541e+09,6383637
3949,69110000,12,2025-12-31,OD,1506,Provisions pour retraite,non lettre,10642561.0,0.0,1.471183e+09,10642561
4041,781000,12,2025-12-31,OD,1502,Transfert de charge,non lettre,0.0,5009280.0,4.220113e+06,-5009280


5-	Identification des écritures au débit de classe 7 : A partir du grand livre, extraire toutes les écritures au débit de la classe 7.

In [ ]:
# ÉTAPE 1 : S'assurer que le numéro de compte est au format texte pour faciliter les extractions 
# Cela permet d'utiliser facilement la méthode .str.startswith()
data['N°compte'] = data['N°compte'].astype(str)

In [ ]:
# ÉTAPE 2 : Filtrer les lignes appartenant à la classe 7 ET ayant un montant au Débit supérieur à 0
condition_classe7_debit = (data['N°compte'].str.startswith('7')) & (data['Débit'] > 0)


In [ ]:
ecritures_debit_classe7 = data[condition_classe7_debit]

In [ ]:
if ecritures_debit_classe7.empty:
    print("Aucune écriture au débit d'un compte de classe 7 n'a été trouvée.")
else:
    print(f"Attention : {len(ecritures_debit_classe7)} écriture(s) au débit de la classe 7 trouvée(s) !")
    display(ecritures_debit_classe7)

Attention : 1 écriture(s) au débit de la classe 7 trouvée(s) !


,N°compte,Mois,Date,Code journaux,N° pièces,libellés,Lettrage,Débit,Crédit,Solde cumulé,Solde au 31 12 2025
4025,70641000,3,2025-03-25,VTE,378,Contre passation,non lettre,55368674.0,0.0,268786531.0,55368674


6-	Identification des écritures qui contiennent chiffre constant : A partir du fichier du détail sur l’ensemble des écritures, nous extraire les écritures au débit ou au crédit qui se terminent par « 000000 ». « 999 ».



In [ ]:
# ÉTAPE 1 : Convertir les colonnes Débit et Crédit en chaînes de caractères pour analyser les derniers chiffres
# On supprime les décimales superflues (ex: ".0") pour ne cibler que les chiffres entiers
debit_str = data['Débit'].astype(str).str.split('.').str[0]
credit_str = data['Crédit'].astype(str).str.split('.').str[0]

In [ ]:
# ÉTAPE 2 : Filtrer les lignes dont le Débit ou le Crédit se termine par "000000" ou "999"
# Note : S'assurer que le montant est supérieur à 0 pour éviter d'attraper les valeurs nulles "0"
condition_chiffre_constant = (
    ((debit_str.str.endswith('000000')) & (data['Débit'] > 0)) |
    ((debit_str.str.endswith('999')) & (data['Débit'] > 0)) |
    ((credit_str.str.endswith('000000')) & (data['Crédit'] > 0)) |
    ((credit_str.str.endswith('999')) & (data['Crédit'] > 0))
)

In [ ]:
ecritures_chiffre_constant = data[condition_chiffre_constant]

In [ ]:
# ÉTAPE 3 : Affichage du résultat
if ecritures_chiffre_constant.empty:
    print("Aucune écriture avec un chiffre constant (finissant par 000000 ou 999) n'a été trouvée.")
else:
    print(f"Attention : {len(ecritures_chiffre_constant)} écriture(s) avec des chiffres constants trouvée(s) !")
    display(ecritures_chiffre_constant)

Attention : 82 écriture(s) avec des chiffres constants trouvée(s) !


,N°compte,Mois,Date,Code journaux,N° pièces,libellés,Lettrage,Débit,Crédit,Solde cumulé,Solde au 31 12 2025
0,10130000,1,2025-01-01,RAN,1,A.N. au 010125,non lettre,0.0,200000000.0,-2.000000e+08,-200000000
128,401100,3,2025-03-13,ACH,295,Réadaptation spot,N,0.0,2000000.0,-1.139435e+08,-2000000
129,401100,3,2025-03-17,BQ1,296,Virement,N,2000000.0,0.0,-1.119435e+08,2000000
508,41110100,1,2025-01-09,BQ1,6,Remise de chèque,non lettre,0.0,12000000.0,-1.666486e+08,-12000000
521,41110100,3,2025-03-18,BQ1,291,Remise de chèque,non lettre,0.0,20000000.0,4.479714e+07,-20000000
...,...,...,...,...,...,...,...,...,...,...,...
2543,585000,1,2025-01-06,CAIS,92,Appro caisse,A,0.0,3000000.0,2.919337e+07,-3000000
2552,585000,1,2025-01-29,BQ1,73,Virement,A,3000000.0,0.0,3.219337e+07,3000000
3221,62720000,12,2025-12-24,ACH,1555,Facture n°2025-0434,non lettre,1000000.0,0.0,1.190507e+09,1000000
3657,6324000,4,2025-04-11,ACH,417,FACTURE N° SNL0100198/2025-04,non lettre,4000000.0,0.0,1.234342e+09,4000000


7-	Identification des écritures extournées : A partir du détail des écritures, sélectionner les écritures qui sont passées entre le 01/01/2026 et le 30/01/2026. Investiguer sur les extournes non justifiées

In [ ]:
# ÉTAPE 1 : S'assurer que la colonne 'Date' est bien au format datetime
data1['Date'] = pd.to_datetime(data1['Date'])

In [ ]:
# ÉTAPE 2 : Filtrer les écritures passées en janvier 2026 (du 01/01/2026 au 30/01/2026)
ecritures_janvier = data1[(data1['Date'] >= '2026-01-01') & (data1['Date'] <= '2026-01-30')]

In [ ]:
# ÉTAPE 3 : Affichage du résultat
if ecritures_janvier.empty:
    print("Aucune écriture trouvée en janvier 2026.")
else:
    print(f"Attention : {len(ecritures_janvier)} écriture(s) trouvée(s) en janvier 2026 (à analyser pour les extournes) :")
    display(ecritures_janvier)

Attention : 372 écriture(s) trouvée(s) en janvier 2026 (à analyser pour les extournes) :


,N°compte,Date,Code journaux,N° pièces,libellés,Lettrage,Débit,Crédit,Solde cumulé
0,10130000,2026-01-01,RAN,1,AN 010126,NaN,NaN,200000000.0,-200000000.0
1,1110100,2026-01-01,RAN,1,AN 010126,NaN,NaN,668605.0,-668605.0
2,12000000,2026-01-01,RAN,1,AN 010126,NaN,NaN,4196400.0,-4196400.0
3,1301000,2026-01-01,RAN,1,AN 010126,NaN,NaN,32193370.0,-32193370.0
4,1961000,2026-01-01,RAN,1,AN 010126,NaN,NaN,10642561.0,-10642561.0
...,...,...,...,...,...,...,...,...,...
367,41110100,2026-01-30,BQ1,144,Remise de chèque,NaN,NaN,31351198.0,609863283.0
368,5211000,2026-01-30,BQ1,144,Remise de chèque,NaN,31351198.0,NaN,208683613.0
369,5211000,2026-01-30,BQ1,145,Compense,NaN,NaN,100.0,208683513.0
370,63180000,2026-01-30,BQ1,145,Compense,NaN,100.0,NaN,153933.0



8-	Identification des écritures passées le week-end : Identifier toutes les écritures passer les week-end et jour férié. 




In [ ]:
# ÉTAPE 1 : vérifier si c'est dans le bon format
data['Date'] = pd.to_datetime(data['Date'])

In [ ]:
# ÉTAPE 2 : Définition des jours fériés légaux au Sénégal pour l'année 2025
jours_feries_2025 = [
    '2025-01-01', # Jour de l'An
    '2025-03-31', # Korité (Aïd el-Fitr)
    '2025-04-04', # Fête de l'Indépendance du Sénégal
    '2025-04-21', # Lundi de Pâques
    '2025-05-01', # Fête du Travail
    '2025-05-29', # Ascension
    '2025-06-07', # Tabaski (Aïd el-Kébir)
    '2025-06-09', # Lundi de Pentecôte
    '2025-09-05', # Gamou / Maouloud (Naissance du Prophète) 
    '2025-08-15', # Assomption
    '2025-11-01', # Toussaint
    '2025-12-25'  # Noël
]

In [ ]:
# ÉTAPE 3 : Convertir la liste des jours fériés en objets datetime (sans les heures pour comparer uniquement les dates)
jours_feries_dt = pd.to_datetime(jours_feries_2025).normalize()

In [ ]:
# ÉTAPE 4 : Extraction du jour de la semaine (0 = Lundi, ..., 5 = Samedi, 6 = Dimanche)
data['Jour_semaine'] = data['Date'].dt.dayofweek

In [ ]:
# Normaliser la colonne date pour éviter les problèmes de comparaison avec l'heure
dates_uniques = data['Date'].dt.normalize()

In [ ]:
# 5. Condition : Soit c'est un week-end (Samedi=5, Dimanche=6), soit la date est dans les jours fériés
condition_weekend_ou_ferie = (data['Jour_semaine'].isin([5, 6])) | (dates_uniques.isin(jours_feries_dt))


In [ ]:
# 6. Filtrage des écritures
ecritures_weekend_ou_feries = data[condition_weekend_ou_ferie]

In [ ]:
# 7. Affichage du résultat
if ecritures_weekend_ou_feries.empty:
    print("Aucune écriture passée un week-end ou un jour férié !")
else:
    print(f"Attention : {len(ecritures_weekend_ou_feries)} écriture(s) passée(s) un week-end ou un jour férié trouvée(s) !")
    display(ecritures_weekend_ou_feries)

Attention : 288 écriture(s) passée(s) un week-end ou un jour férié trouvée(s) !


,N°compte,Mois,Date,Code journaux,N° pièces,libellés,Lettrage,Débit,Crédit,Solde cumulé,Solde au 31 12 2025,Jour_semaine
0,10130000,1,2025-01-01,RAN,1,A.N. au 010125,NaN,NaN,200000000.0,-200000000.0,-200000000,2
3,12900000,1,2025-01-01,RAN,1,A.N. au 010125,NaN,1821048.0,NaN,1821048.0,1821048,2
5,1301000,1,2025-01-01,RAN,2,Résultat de l'exercice,NaN,NaN,6686053.0,-6686053.0,-6686053,2
7,1961000,1,2025-01-01,RAN,1,A.N. au 010125,NaN,NaN,9220113.0,-9220113.0,-9220113,2
10,2441000,1,2025-01-01,RAN,1,A.N. au 010125,NaN,938610.0,NaN,938610.0,938610,2
...,...,...,...,...,...,...,...,...,...,...,...,...
3986,70110100,10,2025-10-05,VTE,1277,Facture n° 25035,NaN,NaN,26450743.0,-898226600.0,-26450743,6
3987,70110100,10,2025-10-05,VTE,1278,Facture n° 25036,NaN,NaN,36345784.0,-934572384.0,-36345784,6
3988,70110100,10,2025-10-05,VTE,1279,Facture n° 25037,NaN,NaN,25200000.0,-959772384.0,-25200000,6
4009,701101100,5,2025-05-29,VTE,644,Facture n° 25020,NaN,NaN,2640000.0,-17380179.0,-2640000,3
